# veritract — verified extraction demo

**Verified extraction: truth-anchored LLM structured output.**

### LLMs hallucinate plausible-looking JSON. veritract adds a **grounding layer** that traces every extracted value back to a character span in the source, giving you the **provenance** - a per-field trust signal, rather than an all-or-nothing response.

---

### Core API

```python
# Bundled: extraction + grounding in one call
result = extract(text, schema, llm, mode="fuzzy")

# Or separately — run LLM once, apply different grounding strategies without re-inference
raw    = extract_raw(text, schema, llm)          # LLM call only → RawExtractionResult
result = ground(raw, llm, mode="fuzzy")          # grounding only → ExtractionResult
```

### Pipeline

```
(optional) Prompt optimisation
         ↓
 Extract_raw — LLM call with GBNF constrained decoding
         ↓
 Ground — fuzzy match → (optional) LLM re-verification
         ↓
 Typed provenance per field  +  quarantine
```

**GBNF (GGML Backus-Naur Form)** is the grammar format used by
[llama.cpp](https://github.com/ggerganov/llama.cpp) under Ollama.
When a JSON schema is provided it is compiled to a GBNF grammar; invalid tokens
are masked to zero probability at every decoding step — the model physically
cannot produce malformed JSON or omit required fields.

**Local inference via [Ollama](https://ollama.com):** runs entirely on your machine.
No API key, no data leaving your network. Supports Metal (Mac), CUDA (Linux/Windows), CPU.

### Input formats
| Format | API |
|---|---|
| Plain text / markdown | `extract(text, schema, llm)` |
| PDF (text + tables + OCR) | `extract_pdf(path, schema, llm)` |
| Images / figures | `extract(caption, schema, llm, images=[b64])` |
| HTML | under development |


## Setup

**Running this notebook:**
1. Make sure [Ollama](https://ollama.com) is running: `ollama serve`
2. Pull the model: `ollama pull gemma4:e4b`
3. Select the **veritract (venv)** kernel (Kernel menu → Change Kernel)

The venv has all dependencies installed including `docling` for PDF and figure support.
To install in a fresh environment (not yet on PyPI):
```bash
git clone https://github.com/LanGuo/veritract.git
cd veritract
python3 -m venv venv
source venv/bin/activate        # Windows: venv\Scripts\activate
pip install -e '.[pdf]'
```
Then register the venv as a Jupyter kernel so you can select it above:
```bash
pip install ipykernel
python3 -m ipykernel install --user --name veritract --display-name "veritract (venv)"
```


In [1]:
import textwrap, json
from veritract import extract, extract_raw, ground, LLMClient, MockLLM, load_images_b64

llm = LLMClient(model="gemma4:e4b", temperature=0.0, seed=42)

# Clinical trial abstract used throughout the demo
ABSTRACT = """\
A double-blind randomised controlled trial evaluated the efficacy of atorvastatin
40 mg daily versus placebo in 486 patients with moderate hypercholesterolaemia.
Participants were followed for 52 weeks. The primary endpoint was reduction in
LDL-cholesterol from baseline, which decreased by 43% in the atorvastatin group
compared to 4% in the placebo group (p < 0.0001). Treatment was well tolerated;
elevated liver enzymes were recorded in 3 patients (0.6%).""".strip()

/Users/languo/src/veritract/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## Part 1 — Define your schema

The schema is standard **JSON Schema**. Each property is a field you want extracted.
All `required` fields always appear in the output — GBNF constrained decoding forces
the model to emit every field. If the model is uncertain it returns an empty string,
which surfaces as quarantined rather than silently missing.

```python
schema = {
    "type": "object",
    "properties": {
        "field_name": {"type": "string"},   # add as many fields as you need
        ...
    },
    "required": ["field_name", ...],        # all required fields always emitted
}
```

In [2]:
SCHEMA = {
    "type": "object",
    "properties": {
        "drug":            {"type": "string"},
        "sample_size":     {"type": "string"},
        "primary_outcome": {"type": "string"},
    },
    "required": ["drug", "sample_size", "primary_outcome"],
}

print("Schema:")
print(json.dumps(SCHEMA, indent=2))

Schema:
{
  "type": "object",
  "properties": {
    "drug": {
      "type": "string"
    },
    "sample_size": {
      "type": "string"
    },
    "primary_outcome": {
      "type": "string"
    }
  },
  "required": [
    "drug",
    "sample_size",
    "primary_outcome"
  ]
}


---
## Part 2 — Write your prompt and extract

The prompt describes **what each field means** and how to extract it.
Key rules to always include:
- Copy verbatim phrases — no paraphrasing or synthesis
- Return an empty string if a field isn't present (not "N/A" or "not found")
- Don't use prior knowledge — extract only from the provided text

If you omit `prompt`, veritract generates a default one from the schema field names.

In [3]:
PROMPT = """\
You are extracting facts from a clinical trial abstract.
Rules:
- Copy the exact verbatim phrase from the text. Do not paraphrase or use prior knowledge.
- If a field is not present in this excerpt, return an empty string.

Fields:
* drug: Drug name and dose as stated (e.g. "atorvastatin 40 mg daily").
* sample_size: Number of patients enrolled as stated.
* primary_outcome: Primary endpoint or outcome measure as stated."""

# mode="fuzzy": fuzzy grounding only — fast, no second LLM call.
# mode="full":  also runs LLM re-verification on quarantined fields (higher recall, ~2× slower).
result = extract(ABSTRACT, SCHEMA, llm, prompt=PROMPT, mode="fuzzy")

print("Extracted fields:")
for field, gf in result.extracted.items():
    ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
    print(f"  [{ptype:>12}]  {field}: {gf['value']!r}")

if result.quarantined:
    print("\nQuarantined:")
    for q in result.quarantined:
        print(f"  {q['field_name']}: {q['value']!r} — {q['reason']}")

Extracted fields:
  [      direct]  drug: 'atorvastatin 40 mg daily'
  [      direct]  sample_size: '486'
  [      direct]  primary_outcome: 'reduction in LDL-cholesterol from baseline'


In [4]:
result.extracted

{'drug': {'value': 'atorvastatin 40 mg daily',
  'span': {'doc_id': None,
   'source_type': 'text',
   'char_start': 69,
   'char_end': 93,
   'text': 'atorvastatin\n40 mg daily',
   'provenance_type': 'direct'},
  'confidence': 100.0},
 'sample_size': {'value': '486',
  'span': {'doc_id': None,
   'source_type': 'text',
   'char_start': 112,
   'char_end': 115,
   'text': '486',
   'provenance_type': 'direct'},
  'confidence': 100.0},
 'primary_outcome': {'value': 'reduction in LDL-cholesterol from baseline',
  'span': {'doc_id': None,
   'source_type': 'text',
   'char_start': 228,
   'char_end': 270,
   'text': 'reduction in\nLDL-cholesterol from baseline',
   'provenance_type': 'direct'},
  'confidence': 100.0}}

In [5]:
# Every grounded field has a precise character span — shown as colored highlight
from IPython.display import HTML, display

_COLORS = {"direct": "#c8f7c5", "paraphrased": "#c5e8f7", "inferred": "#f7f0c5"}

def highlight_html(text, field_name, result):
    gf = result.extracted.get(field_name)
    if not gf or not gf["span"]:
        return f"<p><b>{field_name}:</b> <em>(no span)</em></p>"
    s, e = gf["span"]["char_start"], gf["span"]["char_end"]
    ptype = gf["span"]["provenance_type"]
    color = _COLORS.get(ptype, "#f7ddc5")
    body = (
        text[:s]
        + f'<mark style="background:{color};border-radius:3px;padding:0 2px">'
        + text[s:e] + "</mark>"
        + text[e:]
    ).replace("\n", " ")
    return f"<p><b>{field_name}</b> <code>[{ptype}]</code>: {body}</p>"

display(HTML(
    "<div style='font-family:sans-serif;line-height:1.9;font-size:14px'>"
    + "".join(highlight_html(ABSTRACT, f, result) for f in result.extracted)
    + "</div>"
))

---
## Part 3 — Optional: prompt optimization and few-shot examples

### Prompt optimization

`optimize_prompt` runs iterative refinement: extract → score by grounding rate →
ask the LLM to suggest improvements → repeat. Returns the highest-scoring prompt.

Pass `ground_truth` labels for supervised accuracy scoring.
Benchmarks show **10–20 pp accuracy gains** on clinical NLP tasks after 3–5 iterations.

> **Note:** each iteration calls the LLM twice (extract + refine), so this cell
> takes ~1–2 minutes with `n_iter=2`. In production, run once offline and reuse the result.

In [6]:
from veritract import optimize_prompt

TRAIN_EXAMPLES = [
    {"text": ABSTRACT},
    {"text": (
        "A placebo-controlled study of rosuvastatin 20mg enrolled 312 patients "
        "with dyslipidaemia. The primary endpoint was LDL-C change from baseline at 24 weeks."
    )},
]

# Unsupervised: score by grounding rate — no labels needed
best_prompt = optimize_prompt(TRAIN_EXAMPLES, SCHEMA, llm, n_iter=2)

# Diff: show what changed vs the original prompt
orig_lines = set(PROMPT.strip().splitlines())
new_lines  = set(best_prompt.strip().splitlines())
added   = new_lines - orig_lines
removed = orig_lines - new_lines
print("Changes from original prompt:")
for line in sorted(removed): print(f"  - {line}")
for line in sorted(added):   print(f"  + {line}")

result_opt = extract(ABSTRACT, SCHEMA, llm, prompt=best_prompt, mode="fuzzy")
print("\nWith optimised prompt:")
for field, gf in result_opt.extracted.items():
    ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
    print(f"  [{ptype:>12}]  {field}: {gf['value']!r}")

Changes from original prompt:
  - * drug: Drug name and dose as stated (e.g. "atorvastatin 40 mg daily").
  - * primary_outcome: Primary endpoint or outcome measure as stated.
  - * sample_size: Number of patients enrolled as stated.
  - - Copy the exact verbatim phrase from the text. Do not paraphrase or use prior knowledge.
  - - If a field is not present in this excerpt, return an empty string.
  - Fields:
  - Rules:
  - You are extracting facts from a clinical trial abstract.
  + **Example:**
  + **Extraction:**
  + **Instructions:**
  + **Source Text:**
  + 1. **Verbatim Extraction:** You must copy the exact, verbatim phrase from the source text for each field. Do not paraphrase, abbreviate, synthesize, or modify the text in any way.
  + 2. **Missing Fields:** If a field is not present in the text, you must return an empty string (`""`).
  + 3. **Output Format:** You must return a single JSON object containing exactly the following fields: `drug`, `sample_size`, and `primary_outco


### Few-shot examples

Pass `examples` to show the model what good extractions look like.
Useful when field boundaries are ambiguous or the schema is non-obvious.

In [7]:
EXAMPLES = [{
    "text": (
        "A randomised trial of metformin 500mg twice daily enrolled 120 patients "
        "with type 2 diabetes. The primary outcome was HbA1c reduction at 6 months."
    ),
    "fields": {
        "drug":            "metformin 500mg twice daily",
        "sample_size":     "120 patients",
        "primary_outcome": "HbA1c reduction at 6 months",
    },
}]

result_ex = extract(ABSTRACT, SCHEMA, llm, prompt=PROMPT, examples=EXAMPLES, mode="fuzzy")
print("With examples:")
for field, gf in result_ex.extracted.items():
    ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
    print(f"  [{ptype:>12}]  {field}: {gf['value']!r}")

With examples:
  [      direct]  drug: 'atorvastatin 40 mg daily'
  [      direct]  sample_size: '486'
  [      direct]  primary_outcome: 'reduction in LDL-cholesterol from baseline'


---
## Part 4 — The grounding layer

After `extract_raw`, every field value is verified against the source text.

### From silent gap to auditable signal

Tools like LangExtract use QA-span format: the model answers each field as a
reading-comprehension question. When it can't find an answer it simply skips
the field — you get a smaller dict with **no indication** that a field was
attempted and missed.

GBNF forces **every required field** to appear in the LLM output. The model must
either copy a verbatim phrase or emit an empty string. **veritract catches empty
strings (or values it can't verify) and surfaces them as explicit quarantines.**

### Grounding strategies

| Mode | Grounding | LLM re-verify | When to use |
|---|---|---|---|
| `"no-grounding"` | ✗ | ✗ | Benchmarking raw LLM output |
| `"fuzzy"` | ✓ fuzzy | ✗ | Faster; ~half the LLM calls |
| `"full"` (default) | ✓ fuzzy | ✓ on quarantined | Highest fidelity |

**Fuzzy matching** (rapidfuzz `token_set_ratio`):
- Exact / whitespace-flexible match → precise character span, provenance `direct`
- Fuzzy window match (3-sentence window) → broader span, provenance `paraphrased`
- Below threshold → quarantined

**LLM re-verification** (`mode="full"` only): for fields fuzzy matching couldn't
verify, asks the LLM "does this value appear in the source?". Confirmed → `inferred`.

**Separating `extract_raw` and `ground`** means you pay for LLM inference once
and can apply different grounding strategies — or compare them — without re-running
the model.

### Provenance types

| Type | Meaning |
|---|---|
| `direct` | Exact match — value appears verbatim in source (including across line breaks) |
| `paraphrased` | Fuzzy match above threshold |
| `inferred` | LLM confirmed; fuzzy matching failed (abbreviation, unit conversion) |
| quarantined | Neither could verify — surfaced for human review |

The code cell below runs both tools on the same abstract so you can compare.


In [10]:
import langextract as lx
from langextract.data import ExampleData, Extraction
import logging

# Real EBM-NLP abstract — dental instrument study (source: PMID 10589810).
# "drug" here is a dental instrument, not a pharmaceutical — a genuinely ambiguous field.
PERIO_ABSTRACT = (
    "Clinical effects of root instrumentation using conventional steel or "
    "non-tooth substance removing plastic curettes during supportive periodontal therapy. "
    "40 subjects participated in this parallel, randomized, double-blind study. "
    "The results showed no statistically significant differences between the 2 treatment "
    "modalities regarding bleeding on probing (BOP) and probing pocket depth (PPD)."
)
PERIO_SCHEMA = {
    "type": "object",
    "properties": {
        "drug":            {"type": "string"},
        "sample_size":     {"type": "string"},
        "primary_outcome": {"type": "string"},
    },
    "required": ["drug", "sample_size", "primary_outcome"],
}
PERIO_PROMPT = """You are extracting facts from a clinical trial abstract.
Rules:
- Copy the exact verbatim phrase from the text. Do not paraphrase or use prior knowledge.
- If a field is not present in this excerpt, return an empty string.

Fields:
* drug: Drug name and dose as stated, or intervention/treatment used.
* sample_size: Number of patients enrolled as stated.
* primary_outcome: Primary endpoint or outcome measure as stated."""

# ── LangExtract: no type constraints, all-or-nothing per chunk ─────────────────
# Required: at least one labelled example.
lx_example = ExampleData(
    text=(
        "A randomised trial of metformin 500mg enrolled 120 patients. "
        "The primary outcome was HbA1c reduction at 6 months."
    ),
    extractions=[
        Extraction(extraction_class="drug",            extraction_text="metformin 500mg"),
        Extraction(extraction_class="sample_size",     extraction_text="120 patients"),
        Extraction(extraction_class="primary_outcome", extraction_text="HbA1c reduction at 6 months"),
    ],
)

# Capture the raw model response before LangExtract post-processes it
_raw_response = {}
_orig_resolver = None
try:
    import langextract.resolver as _resolver_mod
    _OrigResolver = _resolver_mod.Resolver
    class _CapturingResolver(_OrigResolver):
        def resolve(self, input_text, *args, **kwargs):
            try:
                parsed = self.format_handler.parse_output(input_text, strict=False)
                _raw_response['parsed'] = parsed
            except Exception:
                _raw_response['raw'] = input_text
            return super().resolve(input_text, *args, **kwargs)
    _resolver_mod.Resolver = _CapturingResolver
except Exception:
    pass

# Suppress the schema-error warning so it doesn't clutter notebook output
logging.getLogger('absl').setLevel(logging.ERROR)

lx_docs = lx.extract(
    text_or_documents=PERIO_ABSTRACT,
    model_id="gemma4:e4b",
    prompt_description=(
        "Extract drug (intervention or treatment used), "
        "sample_size (patients enrolled), "
        "primary_outcome (primary endpoint or outcome measure)."
    ),
    examples=[lx_example],
    show_progress=False,
    temperature=0.0,
)

# Restore log level
logging.getLogger('absl').setLevel(logging.WARNING)

lx_doc = lx_docs[0] if isinstance(lx_docs, list) else lx_docs
lx_fields = {
    ext.extraction_class: ext.extraction_text
    for ext in (lx_doc.extractions if lx_doc and lx_doc.extractions else [])
}

print("LangExtract — raw model JSON output:")
if _raw_response.get('parsed'):
    for item in _raw_response['parsed']:
        print(f"  {item}")
print()
print("LangExtract output (after post-processing):")
for field in PERIO_SCHEMA["required"]:
    val = lx_fields.get(field)
    if val:
        print(f"  {field}: {val!r}")
    else:
        print(f"  {field}: (missing — silent gap)")
print()
print("Why? The model returned null for 'drug' (valid JSON, no grammar to prevent it).")
print("LangExtract discards the ENTIRE extraction when any field type is wrong,")
print("so sample_size and primary_outcome — which were correctly extracted — are lost too.")

print()

# ── veritract: GBNF enforces string type, per-field processing ────────────────
# GBNF grammar bans null for required string fields — the model must emit a string.
# Each field is verified independently; one uncertain field never affects others.
vt_result = extract(PERIO_ABSTRACT, PERIO_SCHEMA, llm, prompt=PERIO_PROMPT, mode="fuzzy")
print("veritract output (GBNF constrained, per-field grounding):")
for field in PERIO_SCHEMA["required"]:
    if field in vt_result.extracted:
        gf = vt_result.extracted[field]
        ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
        print(f"  [{ptype:>12}]  {field}: {gf['value']!r}")
for q in vt_result.quarantined:
    val_repr = repr(q["value"]) if q["value"] else "(empty string — model uncertain)"
    print(f"  [   QUARANTINE]  {q['field_name']}: {val_repr}  ← auditable signal")


LangExtract — raw model JSON output:
  {'drug': None, 'drug_attributes': {}}
  {'sample_size': '40 subjects', 'sample_size_attributes': {}}
  {'primary_outcome': 'bleeding on probing (BOP) and probing pocket depth (PPD)', 'primary_outcome_attributes': {}}

LangExtract output (after post-processing):
  drug: (missing — silent gap)
  sample_size: (missing — silent gap)
  primary_outcome: (missing — silent gap)

Why? The model returned null for 'drug' (valid JSON, no grammar to prevent it).
LangExtract discards the ENTIRE extraction when any field type is wrong,
so sample_size and primary_outcome — which were correctly extracted — are lost too.

veritract output (GBNF constrained, per-field grounding):
  [      direct]  drug: 'conventional steel or non-tooth substance removing plastic curettes'
  [      direct]  sample_size: '40'
  [      direct]  primary_outcome: 'bleeding on probing (BOP) and probing pocket depth (PPD'


---
## Benchmark — vs LangExtract on EBM-NLP 2.0

Evaluated on **155 expert-annotated clinical trial abstracts** (fields: `drug`, `sample_size`, `outcome`).
Both systems: `gemma4:e4b` via Ollama, 3 runs × 155 samples, temp=0.3, optimised prompt.

| | veritract | LangExtract |
|---|---|---|
| **Field accuracy** | **87.7% ± 0.3%** | 84.7% ± 1.9% |
| Extraction latency | 28.9s | **5.5s** |
| Verification cost | +5.4s | +0.2s |
| Grounded (has span) | **90.3%** | 85.7% |
| Quarantined | 9.7% | 0%* |
| Missing (never extracted) | **0%** | ~14.3% |
| Quarantine precision | **100%** | — |
| Quarantine recall | 79% | — |

*LangExtract skips fields it can't answer; veritract always emits every schema field.

**Why is veritract slower?** GBNF constrained decoding enforces grammar validity at
every token step, which is inherently slower than LangExtract's QA-span format
(which just copies a verbatim phrase with no grammar checking). The latency tradeoff
buys: guaranteed JSON validity, mandatory emission of all schema fields, and
richer provenance.

---
## Part 5 — PDF extraction

`extract_pdf` converts PDF → markdown via docling (handles text, tables, scanned
pages via OCR), chunks the markdown, extracts from each chunk independently,
merges results (preferring compact, high-frequency values across chunks), and
grounds against the full document.

In [11]:
from veritract import extract_pdf

TECH_SCHEMA = {
    "type": "object",
    "properties": {
        "architecture_type":       {"type": "string"},
        "total_parameters":        {"type": "string"},
        "context_length":          {"type": "string"},
        "pretraining_token_count": {"type": "string"},
        "alignment_and_rl_method": {"type": "string"},
    },
    "required": [
        "architecture_type", "total_parameters", "context_length",
        "pretraining_token_count", "alignment_and_rl_method",
    ],
}

PDF_PROMPT = """\
You are extracting facts from an LLM technical report.
Rules:
- Copy the exact verbatim phrase from the text. Do not paraphrase or use prior knowledge.
- If a field is not present in this specific excerpt, return an empty string.

Fields:
* architecture_type: Model architecture family and variant (dense, MoE, attention type).
* total_parameters: Total parameter count as stated.
* context_length: Maximum context window as stated.
* pretraining_token_count: Total pretraining tokens as stated.
* alignment_and_rl_method: Post-SFT alignment method (RLHF, DPO, GRPO, BOND, etc.)."""

# Download: https://arxiv.org/pdf/2503.19786  (Gemma 3 technical report)
PDF_PATH = "/tmp/llm_reports/gemma3.pdf"

print("Extracting from Gemma 3 technical report...")
result_pdf = extract_pdf(
    PDF_PATH, TECH_SCHEMA, llm,
    chunk_size=8000, chunk_overlap=500,
    mode="fuzzy", prompt=PDF_PROMPT,
)

print("\nGrounded fields:")
for field, gf in result_pdf.extracted.items():
    if gf["value"]:
        ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
        print(f"  [{ptype:>12}]  {field}: {gf['value'][:90]}")

real_q = [q for q in result_pdf.quarantined if q["value"]]
if real_q:
    print("\nQuarantined:")
    for q in real_q:
        print(f"  {q['field_name']}: {q['value'][:60]!r}")

Extracting from Gemma 3 technical report...


[INFO] 2026-04-30 23:37:45,848 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-30 23:37:45,858 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-30 23:37:45,889 [RapidOCR] download_file.py:60: File exists and is valid: /Users/languo/src/veritract/venv/lib/python3.14/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-30 23:37:45,890 [RapidOCR] main.py:50: Using /Users/languo/src/veritract/venv/lib/python3.14/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-30 23:37:47,521 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-30 23:37:47,523 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-30 23:37:47,525 [RapidOCR] download_file.py:60: File exists and is valid: /Users/languo/src/veritract/venv/lib/python3.14/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-04-30 23:37:47,526 [RapidOCR] main.py:50: Using /Users/languo/src/veritract/venv/lib/python3.14/site-package


Grounded fields:
  [      direct]  architecture_type: decoder-only transformer
  [      direct]  total_parameters: 25,600M
  [      direct]  context_length: 32K and 128K sequence lengths
  [      direct]  pretraining_token_count: 14T tokens for Gemma 3 27B, 12T for the 12B version, 4T for the 4B, and 2T tokens for the 
  [      direct]  alignment_and_rl_method: RL finetuning phase based on improved versions of BOND (Sessa et al., 2024), WARM (Ramé et


---
## Part 6 — Multimodal: figure + paper text

Pass images alongside text using `images=[b64_string]`.
veritract prepends an image-first preamble automatically so the model attends
to the visual before reading the extraction instructions.

This example extracts from **Figure 2** of the Gemma 3 technical report:
a radar chart comparing Gemma 2 and Gemma 3 across six capability axes.
The schema mixes visual fields (what you see in the chart) and textual fields
(the authors' interpretation in Section 5.1):

| Field | Source |
|---|---|
| `figure_type` | image only — chart style not stated verbatim in text |
| `models_shown` | image labels — model names visible in the chart |
| `evaluation_axes` | both — axis labels in image, listed verbatim in paper text |
| `author_interpretation` | paper text — authors' summary conclusion |
| `focus_capability` | paper text — capability they specifically highlight |

Grounding then tells you which values could be verified against the document
(direct span) and which came purely from vision (quarantined).


In [12]:
import base64, io
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

FIGURE_SCHEMA = {
    "type": "object",
    "properties": {
        "figure_type":           {"type": "string"},  # from image
        "models_shown":          {"type": "string"},  # from image
        "evaluation_axes":       {"type": "string"},  # from image + text
        "author_interpretation": {"type": "string"},  # from paper text
        "focus_capability":      {"type": "string"},  # from paper text
    },
    "required": [
        "figure_type", "models_shown", "evaluation_axes",
        "author_interpretation", "focus_capability",
    ],
}

FIGURE_PROMPT = """\
You are analyzing a figure from an ML research paper alongside its caption and surrounding text.

Rules:
- For visual fields (figure_type, models_shown): read the image and copy labels exactly as shown.
- For text fields (author_interpretation, focus_capability): copy verbatim from the paper text provided.
- For fields in both sources (evaluation_axes): prefer verbatim phrasing from the paper text.
- If a field cannot be determined, return an empty string.

Fields:
* figure_type: Type of chart visible in the image (e.g. radar chart, bar chart, line plot).
* models_shown: Model names visible as labels in the figure, comma-separated as labelled.
* evaluation_axes: Capability categories shown as axes — copy verbatim from paper text if stated there.
* author_interpretation: The authors' overall conclusion about this figure, verbatim from paper text.
* focus_capability: The specific capability the authors emphasize for improvement, verbatim from paper text."""

# Load PDF with image extraction enabled
opts = PdfPipelineOptions()
opts.generate_picture_images = True
opts.images_scale = 2.0
conv = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)})
doc  = conv.convert(PDF_PATH)
full_text = doc.document.export_to_markdown()

# Figure 2 — radar chart comparing Gemma 2 and Gemma 3 across six capabilities
qualifying = [
    (p.get_image(doc.document), p.caption_text(doc.document) or "")
    for p in doc.document.pictures
    if p.get_image(doc.document) and p.get_image(doc.document).width >= 200
]
img, caption = qualifying[1]
print(f"Figure: {img.width}×{img.height}px")
print(f"Caption: {caption[:120]}")

# Surrounding text: the paragraph in Section 5.1 that discusses Figure 2
sec51_start = full_text.find("We use several standard benchmarks as probes")
sec52_start = full_text.find("5.2. Local:Global")
surrounding = full_text[sec51_start:sec52_start].strip()

# Source text seen by the model = caption + surrounding paragraph
# This lets the model draw from both the image (visual) and the paper (textual)
source_for_model = f"{caption}\n\nFrom the paper (Section 5.1):\n{surrounding[:600]}"

buf = io.BytesIO()
img.save(buf, format="PNG")
b64 = base64.b64encode(buf.getvalue()).decode()

# Stage 1: extract — model sees image + caption + surrounding text
raw_fig = extract_raw(
    source_for_model, FIGURE_SCHEMA, llm,
    prompt=FIGURE_PROMPT, images=[b64],
    source_type="figure", doc_id="gemma3.pdf",
)

# Stage 2: ground against the full document
# Values stated in the paper get verified spans; vision-only values are quarantined
raw_fig.source_text = full_text
result_fig = ground(raw_fig, llm=None, mode="fuzzy")

print("\nGrounded (value found in document text — span verified):")
for field, gf in result_fig.extracted.items():
    if gf["value"]:
        ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
        print(f"  [{ptype:>12}]  {field}: {gf['value'][:90]}")

print("\nQuarantined (vision-only — value not found in document text):")
for q in result_fig.quarantined:
    if q["value"]:
        print(f"  {q['field_name']}: {q['value'][:70]!r}")


[INFO] 2026-04-30 23:47:05,578 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-30 23:47:05,587 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-30 23:47:05,620 [RapidOCR] download_file.py:60: File exists and is valid: /Users/languo/src/veritract/venv/lib/python3.14/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-30 23:47:05,620 [RapidOCR] main.py:50: Using /Users/languo/src/veritract/venv/lib/python3.14/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-30 23:47:07,342 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-30 23:47:07,343 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-30 23:47:07,345 [RapidOCR] download_file.py:60: File exists and is valid: /Users/languo/src/veritract/venv/lib/python3.14/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-04-30 23:47:07,345 [RapidOCR] main.py:50: Using /Users/languo/src/veritract/venv/lib/python3.14/site-package

Figure: 952×215px
Caption: Figure 2 | Summary of the performance of different pre-trained models from Gemma 2 and 3 across gene

Extracted:
  [ paraphrased]  best_result: 9B (Gemma 2) in Code (92
  [      direct]  key_finding: Summary of the performance of different pre-trained models from Gemma 2 and 3 across gener

Quarantined (vision-only — not found in document text):
  figure_type: 'Radar chart'
  models_compared: 'Gemma 2, Gemma 3'
  metric: 'Code, Factuality, Reasoning, Multilingual, Science'


---
## Summary

| | veritract | Plain JSON / Instructor |
|---|---|---|
| Extracts structured JSON | ✓ | ✓ |
| Constrained decoding (guaranteed valid JSON) | ✓ GBNF | ✓ (provider-dependent) |
| Every field traced to source span | ✓ | ✗ |
| Hallucinations caught and quarantined | ✓ | ✗ |
| Trust signal per field (not per response) | ✓ | ✗ |
| Fully local / air-gapped | ✓ Ollama | ✗ API required |
| PDF extraction with table + OCR support | ✓ docling | ✗ |
| Multimodal (image + figure) extraction | ✓ | provider-dependent |

**GitHub:** https://github.com/LanGuo/veritract
**Install (from source):** `git clone https://github.com/LanGuo/veritract.git && pip install -e '.[pdf]'`
*(PyPI release coming soon)*